# Week 2: MLP on Fashion-MNIST

In this notebook, I build and train a simple multilayer perceptron (MLP) for Fashion-MNIST image classification. I also compare three learning rates.

## 1. Import packages

In [ ]:
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

PROJECT_ROOT = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                     if (folder / "week1" / "week1_ml_basics.ipynb").is_file()
                     and (folder / "week5" / "week5_vae.ipynb").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter in the research project folder or one of its subfolders.")
DATA_DIR = PROJECT_ROOT / "data"

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

print("PyTorch version:", torch.__version__)
print("Done.")


## 2. Select a device

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)
print("Done.")


## 3. Download and load Fashion-MNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

train_dataset = datasets.FashionMNIST(
    root=str(DATA_DIR), train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST(
    root=str(DATA_DIR), train=False, download=True, transform=transform
)

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))
print("One image shape:", train_dataset[0][0].shape)
print("Done.")


## 4. Create DataLoaders

In [ ]:
batch_size = 128

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

images, labels = next(iter(train_loader))
print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)
print("Done.")


## 5. Define the MLP model

The model structure is 784 → 256 → 128 → 10. The `Flatten` layer changes each 28 × 28 image into 784 features.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.network(x)

torch.manual_seed(seed)
model = MLP().to(device)
print(model)
print("Done.")


## 6. Define the loss function and optimizer

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Loss function:", loss_fn)
print("Optimizer:", optimizer.__class__.__name__)
print("Done.")


## 7. Define one training epoch

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    average_loss = total_loss / total
    accuracy = correct / total
    return average_loss, accuracy

print("Done.")


## 8. Define the evaluation function

In [ ]:
@torch.inference_mode()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = loss_fn(logits, labels)

        total_loss += loss.item() * images.size(0)
        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    average_loss = total_loss / total
    accuracy = correct / total
    return average_loss, accuracy

print("Done.")


## 9. Train the model

The history dictionary stores the values needed for the plots.

In [ ]:
epochs = 5
history = {
    "train_loss": [],
    "train_accuracy": [],
    "test_loss": [],
    "test_accuracy": [],
}

for epoch in range(epochs):
    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, loss_fn, optimizer, device
    )
    test_loss, test_accuracy = evaluate(
        model, test_loader, loss_fn, device
    )

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["test_loss"].append(test_loss)
    history["test_accuracy"].append(test_accuracy)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"train loss: {train_loss:.4f} | "
        f"train accuracy: {train_accuracy:.4f} | "
        f"test accuracy: {test_accuracy:.4f}"
    )


## 10. Plot training loss and test accuracy

In [ ]:
epochs_axis = range(1, epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_axis, history["train_loss"], marker="o", label="Train loss")
axes[0].plot(epochs_axis, history["test_loss"], marker="o", label="Test loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss versus epoch")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_axis, history["train_accuracy"], marker="o", label="Train accuracy")
axes[1].plot(epochs_axis, history["test_accuracy"], marker="o", label="Test accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy versus epoch")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()


## 11. Compare learning rates

Only the learning rate changes in this experiment. The model structure, data, batch size, number of epochs, and random seed stay the same.

In [ ]:
def train_model_for_learning_rate(learning_rate, epochs=5):
    torch.manual_seed(seed)
    experiment_model = MLP().to(device)
    experiment_loss_fn = nn.CrossEntropyLoss()
    experiment_optimizer = torch.optim.Adam(
        experiment_model.parameters(), lr=learning_rate
    )

    experiment_history = {
        "train_loss": [],
        "train_accuracy": [],
        "test_accuracy": [],
    }

    for epoch in range(epochs):
        train_loss, train_accuracy = train_one_epoch(
            experiment_model, train_loader, experiment_loss_fn,
            experiment_optimizer, device
        )
        _, test_accuracy = evaluate(
            experiment_model, test_loader, experiment_loss_fn, device
        )

        experiment_history["train_loss"].append(train_loss)
        experiment_history["train_accuracy"].append(train_accuracy)
        experiment_history["test_accuracy"].append(test_accuracy)

    return experiment_history

learning_rates = [0.1, 0.01, 0.001]
learning_rate_histories = {}

for learning_rate in learning_rates:
    print(f"Training with learning rate = {learning_rate}")
    learning_rate_histories[learning_rate] = train_model_for_learning_rate(
        learning_rate, epochs=5
    )

print("Done.")


## 12. Plot the learning-rate comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for learning_rate, experiment_history in learning_rate_histories.items():
    axes[0].plot(
        range(1, epochs + 1),
        experiment_history["train_loss"],
        marker="o",
        label=f"lr={learning_rate}",
    )
    axes[1].plot(
        range(1, epochs + 1),
        experiment_history["test_accuracy"],
        marker="o",
        label=f"lr={learning_rate}",
    )

axes[0].set_title("Training loss for different learning rates")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Training loss")
axes[0].legend()
axes[0].grid(True)

axes[1].set_title("Test accuracy for different learning rates")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Test accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()


## 13. My observations

In my five-epoch run, `lr=0.1` made training fail: the final test accuracy was about 10%, close to guessing among ten classes. `lr=0.01` reached 85.96%, while `lr=0.001` reached 88.04% and had the lowest final training loss of these three settings. A much smaller learning rate would usually make progress slower, but I did not test one below 0.001 here. I changed only the learning rate so that the differences could be linked to that setting.


## Week 2 concepts in my own words

- **Epoch:** one pass through the full training set.
- **Batch:** a group of samples processed together.
- **Mini-batch:** a small batch used for one parameter update.
- **Learning rate:** how far the optimizer moves on each update.
- **Parameter:** a weight or bias the model learns.
- **Gradient:** how the loss changes as a parameter changes.
- **Backpropagation:** calculating gradients from the loss back through the network.
- **Optimizer:** the method that uses gradients to update parameters.
- **Activation function:** a nonlinear step that lets a network learn more than one linear mapping.
- **Cross-entropy:** a loss for classification that penalizes low probability on the true class.
- **Training mode:** the mode used while learning model parameters.
- **Evaluation mode:** the mode used to check predictions without training updates.

One training iteration is: **data → forward pass → prediction → loss → backward pass → parameter update**. `optimizer.zero_grad()` clears old gradients before the next iteration.
